In [1]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

# Select parent sources who have at least one significant emission line (>5 S/N ratio)

In [3]:
FIT = FitSpectrum()
DP = DP()

# dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data('./catalogs/all_catalog.fits')
ALL_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=None)
ALL_SPECTRA = ALL_SPECTRA.subtype_filter(subtype='QSO', exclude=True)
ALL_SPECTRA = ALL_SPECTRA.stack_data()
ALL_SPECTRA = ALL_SPECTRA.shift_to_rest_frame()
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 99812


In [4]:
ALL_SPECTRA = FIT.label_emission_lines(ALL_SPECTRA, 5)
ALL_SPECTRA = FIT.significant_emission_filter(ALL_SPECTRA)
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 58596


In [5]:
# ALL_SPECTRA = ALL_SPECTRA.shrink_dataset(5)
# print('Number of spectra after shrinking:', len(ALL_SPECTRA.targetID))

In [6]:
dp_parent, model_1comp, left_2comp, right_2comp = DP.fit_all(data_class=ALL_SPECTRA, n_jobs=10)

100%|██████████| 58596/58596 [09:03<00:00, 107.75it/s]


In [7]:
filenames = {
    'all'       : './catalogs/0310_all.fits',
    'dps'       : './catalogs/0310_dps.fits',
    'cs'        : './catalogs/0310_cs.fits',
    'nbcs'      : './catalogs/0310_nbcs.fits',
    'cs-nbcs'   : './catalogs/0310_cs-nbcs.fits'
}

In [8]:
# dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data(filenames['all'])
dp_parent = DP.bpt_classification(dp_parent, sigmas=1/np.sqrt(np.abs(ALL_SPECTRA.data_stack[:, 2, :])), model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp, two_comp=True)
DP.get_catalog(df=dp_parent, fname=filenames['all'], model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)

/var/folders/_b/sl_t4k5539781f29qf723b080000gn/T/ipykernel_31992/1894848205.py:2: RuntimeWarning: divide by zero encountered in divide
  dp_parent = DP.bpt_classification(dp_parent, sigmas=1/np.sqrt(np.abs(ALL_SPECTRA.data_stack[:, 2, :])), model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp, two_comp=True)
/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:353: RuntimeWarning: divide by zero encountered in log10
  nii_halpha = np.log10(line_fluxes[3]/line_fluxes[2])
/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:352: RuntimeWarning: divide by zero encountered in log10
  oiii_hbeta = np.log10(line_fluxes[1]/line_fluxes[0])


In [9]:
dp_sample, model_1comp, left_2comp, right_2comp = DP.select_dp_sample(dp_parent, model_1comp, left_2comp, right_2comp)
DP.get_catalog(df=dp_sample, fname=filenames['dps'], model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)

In [10]:
dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data(filenames['all'])
dps_df, _, _, _ = DP.extract_fits_data(filenames['dps'])
cs_df, nbcs_df, cs_nbcs_df = DP.select_nbcs(dp_parent=dp_parent, dp_sample=dps_df)
cs_df['BPT_2comp'] = 0
nbcs_df['BPT_2comp'] = 0
cs_nbcs_df['BPT_2comp'] = 0
DP.get_catalog(cs_df, model_1comp=model_1comp[cs_df.index], left_2comp=left_2comp[cs_df.index], right_2comp=right_2comp[cs_df.index], fname=filenames['cs'])
DP.get_catalog(nbcs_df, model_1comp=model_1comp[nbcs_df.index], left_2comp=left_2comp[nbcs_df.index], right_2comp=right_2comp[nbcs_df.index], fname=filenames['nbcs'])
DP.get_catalog(cs_nbcs_df, model_1comp=model_1comp[cs_nbcs_df.index], left_2comp=left_2comp[cs_nbcs_df.index], right_2comp=right_2comp[cs_nbcs_df.index], fname=filenames['cs-nbcs'])